<a href="https://colab.research.google.com/github/nilsugungor/potsdam-hackathon/blob/main/2026_llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-4-Maverick-17B-128E-Instruct"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Vision olmayan, saf metin odaklı Meta modeli (En kararlısı)
model_id = "unsloth/llama-3.1-8b-instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("✅ Llama-3.1-8B Hazır! Artık metin üretebiliriz.")

In [ ]:
prompt = """Write a complex paragraph about climate change in 2040.
Use temporal sequences in the sentences.
Make it scientifically grounded but narrative."""

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7)
# Sadece modelin yeni ürettiği kısmı kesip alır
clifi_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

print("\n--- Üretilen İklim Analizi ---\n")
print(clifi_text)